# GPU dimensioning

We can think of the process as a flow of data to be updated, and the optimization is a matter of dimensioning the channels of flow of information so it fits optimally to the capacity of the "channels" through whích we want it to flow.

The input (and output) may be considered fixed (they could be varied only at the cost of reduced advantages in other parts of the code.   So, let's freeze the input as 1 million patches, all of dimension 16x16x16 cells (surrounded by 3 guard zone layers, so storage-wise 22x22x22), with 8 variables and 5 time slices (not to be forgotten).   This is the amount of storage needed:

In [1]:
n=16
ng=3
nw=(n+2*ng)**3
nv=8
nt=5
words_per_timeslice=nv*nw
words_per_patch=nt*words_per_timeslice
bytes_per_word=4
bytes_per_patch=bytes_per_word*words_per_patch
bytes_per_patch

1703680

In [2]:
TB=1024**4
memory=1*TB
patches=memory/bytes_per_patch
patches

645374.4997746056

We conclude that we can store about 0.5 million patches per TB.  Of those about 5% may be assumed to be ready-to-update at any one time, so we can think in terms of 25k patches:

In [3]:
active_patches=0.05*5e5
active_patches

25000.0

The number of active patches and the optimal number of units of work to give to the GPUs should NOT be locked together.  To for example give the GPU 25k tasks to update would be extremely sub-optimal.   Hence we need to think in terms of ___bunching___; i.e. merging the work to be done into much larger blocks of work.

To see how that should be accomplished, we should look at the amount of work and memory involved in updating.   That amount is NOT 5% of 1 TB, but only 1% of 1 TB, since only one of the 5 time slices is going to be updated.   So, the amount of memory to send and receive to the GPUs is about 8 GB:

In [4]:
bytes_to_send=bytes_per_word*active_patches*words_per_timeslice
GB=1024**3
bytes_to_send/GB

7.933378219604492

Inside the GPU, of order 50-100 words per cell are needed as scratch space, so the memory requirement inside the GPUs is of order 50-100 GB:

In [5]:
scratch_per_cell=100
GPU_bytes=bytes_per_word*active_patches*nw*scratch_per_cell
GPU_bytes/GB

99.16722774505615

The unit of work inside the GPUs can be most easily parametrized in terms of the ___bunching___ of a number of tasks into one block of work, containing for example a million cells:

In [6]:
patches_per_bunch=100
work_cells=patches_per_bunch*nw
work_cells/1e6

1.0648

It is of central importance to appreciate that every single multiplication, addition and so on will have this number of items to work on, and that this number _must_ be much larger than the number of cells in a patch.   

The number of cores that work on one block of work, and the number of available cores are related as follows:

In [7]:
GPUs_per_node=4

### A100 parameters

The A100 can handle about 16t threads per core, or up to 4096 warps with 32 threads each:

In [8]:
SM_per_GPU=64
cores_per_GPU=6912
cores_per_SM=cores_per_GPU/SM_per_GPU
warps_per_SM=64
threads_per_warp=32
warps_per_GPU=SM_per_GPU*warps_per_SM
threads_per_SM=threads_per_warp*warps_per_SM
threads_per_GPU=SM_per_GPU*threads_per_SM
cores_per_SM,threads_per_SM,threads_per_GPU,warps_per_GPU,threads_per_GPU/cores_per_GPU

(108.0, 2048, 131072, 4096, 18.962962962962962)

We thus have the task of dimensioning blocks of work so they oversubscribe the amount of cores enough to hide memory latency.  We can parametrize that in an oversubscribe factor:

In [9]:
oversubscribe=16
threads_per_GPU=cores_per_GPU*oversubscribe
threads_per_node=threads_per_GPU*GPUs_per_node
threads_per_node

442368

Hence we need to engage about half a million cells at any one time on the GPUs.  The bunching factor is then

In [10]:
patches_per_bunch=threads_per_node/nw
patches_per_bunch

41.544703230653646

We conclude that we may choose to bunch for example 32 patches together, or fewer if we use n=32.  Let's make a procedure:

In [11]:
def GPU_dimensioning(n=16,ng=3,nv=8,nt=5,GPU='A100',GPUs_per_node=4,bytes_per_word=4,
                     oversubscribe=4,memory_TB=1,scratch_per_cell=100):
    if GPU=='A100':
        SM_per_GPU=64
        cores_per_GPU=6912
        max_warps_per_SM=64
        threads_per_warp=32
        GB_per_GPU=40
    elif GPU=='V100':
        SM_per_GPU=80
        cores_per_GPU=5120
        max_warps_per_SM=32
        threads_per_warp=32
        GB_per_GPU=32
    
    print('The {} GPU has:\n'.format(GPU))
    print('       VRAM per GPU: {:.0f} GB'.format(GB_per_GPU))
    print('      Cores per GPU: {:.0f}'.format(cores_per_GPU))
    print('        SMs per GPU: {:.0f}'.format(SM_per_GPU))
    print('   Max warps per SM: {:.0f}'.format(max_warps_per_SM))
    print('   Threads per warp: {:.0f}'.format(threads_per_warp))

    print('\nWith {} {} GPUs per node and {} TB of memory,'.format(GPUs_per_node,GPU,memory_TB))
    print('using patches with {}**3 cells plus {} guard cells,'.format(n,ng))
    print('and assuming {} scratch words per cell we need,'.format(scratch_per_cell))
    print('for a factor {} oversubscription:\n'.format(oversubscribe))

    nw=(n+2*ng)**3
    words_per_timeslice=nv*nw
    words_per_patch=nt*words_per_timeslice
    bytes_per_patch=bytes_per_word*words_per_patch    
    max_warps_per_GPU=SM_per_GPU*max_warps_per_SM
    max_threads_per_SM=threads_per_warp*max_warps_per_SM
    max_threads_per_GPU=SM_per_GPU*max_threads_per_SM
    
    # Compute max number of patches
    GB=1024**3
    TB=1024**4
    memory=memory_TB*TB
    patches=memory/bytes_per_patch

    # Compute patches per bunch
    threads_per_GPU=cores_per_GPU*oversubscribe
    threads_per_node=threads_per_GPU*GPUs_per_node
    patches_per_bunch=threads_per_node/nw
    threads_per_SM=threads_per_GPU/SM_per_GPU
    warps_per_GPU=threads_per_GPU/threads_per_warp
    warps_per_SM=warps_per_GPU/SM_per_GPU
    warps_per_node=warps_per_SM*SM_per_GPU*GPUs_per_node
    bunches_per_node=patches/patches_per_bunch

    print(' max patches per node: {:.0f}'.format(patches))
    print('    patches per bunch: {:.0f}'.format(patches_per_bunch))
    print('     bunches per node: {:.0f}'.format(bunches_per_node))
    print('       warps per node: {:.0f}'.format(warps_per_node))
    print('     bunches per warp: {:.1f}'.format(bunches_per_node/warps_per_node))
    print('      active fraction: {:.3f}'.format(warps_per_node/bunches_per_node))
    
    active_patches=patches_per_bunch*warps_per_GPU
    GPU_bytes=bytes_per_word*active_patches*nw*scratch_per_cell
    print('    GPU memory needed: {:.0f} GB'.format(GPU_bytes/GB))
    
    print('\nWe then have:\n')
    print('    threads per node: {:.0f}'.format(threads_per_node))
    print('     threads per GPU: {:.0f}'.format(threads_per_GPU))
    print('      threads per SM: {:.0f}'.format(threads_per_SM))
    print('        Warps per SM: {:.0f}'.format(warps_per_SM))

GPU_dimensioning()

The A100 GPU has:

       VRAM per GPU: 40 GB
      Cores per GPU: 6912
        SMs per GPU: 64
   Max warps per SM: 64
   Threads per warp: 32

With 4 A100 GPUs per node and 1 TB of memory,
using patches with 16**3 cells plus 3 guard cells,
and assuming 100 scratch words per cell we need,
for a factor 4 oversubscription:

 max patches per node: 645374
    patches per bunch: 10
     bunches per node: 62138
       warps per node: 3456
     bunches per warp: 18.0
      active fraction: 0.056
    GPU memory needed: 36 GB

We then have:

    threads per node: 110592
     threads per GPU: 27648
      threads per SM: 432
        Warps per SM: 14


In [12]:
GPU_dimensioning(oversubscribe=4,n=16)

The A100 GPU has:

       VRAM per GPU: 40 GB
      Cores per GPU: 6912
        SMs per GPU: 64
   Max warps per SM: 64
   Threads per warp: 32

With 4 A100 GPUs per node and 1 TB of memory,
using patches with 16**3 cells plus 3 guard cells,
and assuming 100 scratch words per cell we need,
for a factor 4 oversubscription:

 max patches per node: 645374
    patches per bunch: 10
     bunches per node: 62138
       warps per node: 3456
     bunches per warp: 18.0
      active fraction: 0.056
    GPU memory needed: 36 GB

We then have:

    threads per node: 110592
     threads per GPU: 27648
      threads per SM: 432
        Warps per SM: 14


In [13]:
GPU_dimensioning(GPU='V100',oversubscribe=4,n=16)

The V100 GPU has:

       VRAM per GPU: 32 GB
      Cores per GPU: 5120
        SMs per GPU: 80
   Max warps per SM: 32
   Threads per warp: 32

With 4 V100 GPUs per node and 1 TB of memory,
using patches with 16**3 cells plus 3 guard cells,
and assuming 100 scratch words per cell we need,
for a factor 4 oversubscription:

 max patches per node: 645374
    patches per bunch: 8
     bunches per node: 83886
       warps per node: 2560
     bunches per warp: 32.8
      active fraction: 0.031
    GPU memory needed: 20 GB

We then have:

    threads per node: 81920
     threads per GPU: 20480
      threads per SM: 256
        Warps per SM: 8
